# Notebook 11 — Polynomial Reduction and Irreducible Polynomials

**Volume 1: Foundations, Finite Fields, and AES**

## Sections
1. Introduction and setup
2. Polynomial representation and display
3. Polynomial long division over GF(2)
4. Toy modulus: folding rules
5. Toy modulus: worked reductions
6. The AES modulus — 0x11B
7. AES folding rule and step-by-step reducer
8. Interactive reduction explorer
9. Checking irreducibility over GF(2)
10. Summary and bridge to Module 12

## Section 1 — Introduction

Polynomial multiplication over GF(2) (covered in Module 10) can produce polynomials of degree 14 or higher when two degree-7 byte-polynomials are multiplied together. But a byte can only represent polynomials up to degree 7.

**Polynomial reduction** is the process of folding the high-degree terms back down using a chosen modulus polynomial — exactly as integer modular arithmetic wraps large numbers around a fixed modulus.

The AES cipher uses the irreducible polynomial:

    m(x) = x^8 + x^4 + x^3 + x + 1   (hex: 0x11B)

Its irreducibility ensures that the 256 byte-polynomials form a proper finite field GF(2^8), where every nonzero element has a multiplicative inverse.

In [ ]:
# Section 2 — Polynomial representation and display
# A polynomial over GF(2) is stored as an integer bitmask.
# Bit d == coefficient of x^d (LSB = constant term).

def poly_to_str(v, max_deg=14):
    """Convert bitmask to human-readable polynomial string."""
    if v == 0:
        return '0'
    terms = []
    for d in range(max_deg, -1, -1):
        if (v >> d) & 1:
            if   d == 0: terms.append('1')
            elif d == 1: terms.append('x')
            else:        terms.append(f'x^{d}')
    return ' + '.join(terms)

def poly_from_coeffs(*coeffs):
    """Build bitmask from list of exponents that have coefficient 1.
    e.g. poly_from_coeffs(8,4,3,1,0) -> 0x11B
    """
    v = 0
    for c in coeffs:
        v |= (1 << c)
    return v

# Demonstrate
AES_MOD = poly_from_coeffs(8, 4, 3, 1, 0)
TOY_MOD = poly_from_coeffs(3, 1, 0)

print(f'AES modulus: {poly_to_str(AES_MOD)}  = 0x{AES_MOD:03X}')
print(f'Toy modulus: {poly_to_str(TOY_MOD)}  = 0x{TOY_MOD:02X}')

In [ ]:
# Section 3 — Polynomial long division over GF(2)
# Returns (quotient, remainder) such that dividend = quotient*divisor + remainder

def poly_degree(v):
    """Degree of polynomial v (returns -1 for the zero polynomial)."""
    if v == 0:
        return -1
    return v.bit_length() - 1

def poly_divmod(dividend, divisor):
    """Polynomial division over GF(2). Returns (quotient, remainder)."""
    if divisor == 0:
        raise ValueError('Division by zero polynomial')
    q = 0
    r = dividend
    deg_d = poly_degree(divisor)
    while poly_degree(r) >= deg_d:
        shift = poly_degree(r) - deg_d
        q ^= (1 << shift)
        r ^= (divisor << shift)
    return q, r

def poly_reduce(val, modulus):
    """Return val mod modulus over GF(2)."""
    _, r = poly_divmod(val, modulus)
    return r

# Quick test
p = poly_from_coeffs(4, 2)  # x^4 + x^2
r = poly_reduce(p, TOY_MOD)
print(f'({poly_to_str(p)}) mod ({poly_to_str(TOY_MOD)}) = {poly_to_str(r)}')
# Expect: x

In [ ]:
# Section 4 — Toy modulus: deriving folding rules
# m(x) = x^3 + x + 1  =>  x^3 ≡ x + 1
# Multiply both sides by x to get higher rules.

print('Folding rules for toy modulus x^3 + x + 1:')
print()
for deg in range(3, 8):
    high = 1 << deg
    reduced = poly_reduce(high, TOY_MOD)
    print(f'  x^{deg}  ≡  {poly_to_str(reduced)}')

In [ ]:
# Section 5 — Toy modulus: worked reductions with step tracing

def toy_reduce_verbose(val):
    """Reduce val mod x^3+x+1, printing each substitution step."""
    FOLD_3 = poly_from_coeffs(1, 0)  # x+1
    cur = val
    print(f'Start: {poly_to_str(cur)}')
    for d in range(14, 2, -1):
        if (cur >> d) & 1:
            # x^d = x^(d-3) * x^3 ≡ x^(d-3) * (x+1)
            fold = (FOLD_3 << (d - 3))
            print(f'  x^{d} → replace with {poly_to_str(fold)}')
            cur ^= (1 << d)
            cur ^= fold
            print(f'  After: {poly_to_str(cur)}')
    print(f'Result: {poly_to_str(cur)}')
    return cur

print('=== Reduce x^4 + x^2 mod x^3+x+1 ===')
toy_reduce_verbose(poly_from_coeffs(4, 2))
print()
print('=== Reduce x^5 + x^2 + x mod x^3+x+1 ===')
toy_reduce_verbose(poly_from_coeffs(5, 2, 1))

In [ ]:
# Section 6 — The AES modulus 0x11B

print('AES modulus details')
print('-------------------')
print(f'Polynomial : {poly_to_str(AES_MOD)}')
print(f'Hex        : 0x{AES_MOD:03X}')
print(f'Decimal    : {AES_MOD}')
print(f'Binary     : {AES_MOD:09b}  (9 bits — x^8 is the 9th bit)')
print()

# The folded form of x^8 is the lower 8 bits of 0x11B
FOLD_8 = AES_MOD ^ (1 << 8)   # remove x^8 term
print(f'Folded form of x^8: {poly_to_str(FOLD_8)}  = 0x{FOLD_8:02X}')
print()
print('Verify: x^8 + (folded form) = AES modulus?')
print(f'  {poly_to_str(1<<8)} + {poly_to_str(FOLD_8)} = {poly_to_str((1<<8)^FOLD_8)}')
print(f'  Equals AES modulus: {((1<<8)^FOLD_8) == AES_MOD}')

In [ ]:
# Section 7 — AES folding rule: step-by-step reducer

def aes_reduce_verbose(val):
    """Reduce val mod AES modulus, printing each folding step."""
    FOLD_8 = 0x1B  # x^4 + x^3 + x + 1
    cur = val
    print(f'Start: {poly_to_str(cur)}  (0x{cur:04X})')
    for d in range(14, 7, -1):
        if (cur >> d) & 1:
            shifted_fold = FOLD_8 << (d - 8)
            print(f'  x^{d} → fold: XOR 0x{shifted_fold:04X}  ({poly_to_str(shifted_fold)})')
            cur ^= (1 << d)
            cur ^= shifted_fold
            print(f'  After: {poly_to_str(cur)}  (0x{cur:04X})')
    result = cur & 0xFF
    print(f'Result: {poly_to_str(result)}  (0x{result:02X})')
    return result

print('=== Reduce x^8 + x^4 + x^2 mod AES modulus ===')
aes_reduce_verbose(poly_from_coeffs(8, 4, 2))
print()
print('=== Reduce x^8 + x^7 + x^4 + x + 1 mod AES modulus ===')
aes_reduce_verbose(poly_from_coeffs(8, 7, 4, 1, 0))

In [ ]:
# Section 8 — Interactive reduction explorer
# Call reduction_explorer(hex_value) to reduce any polynomial.

def reduction_explorer(hex_value, modulus=AES_MOD, modulus_name='AES'):
    """
    Reduce hex_value modulo the given modulus and show every step.
    
    Parameters
    ----------
    hex_value   : int  — the polynomial to reduce (e.g. 0x1A5C)
    modulus     : int  — the modulus polynomial (default: AES 0x11B)
    modulus_name: str  — display name for the modulus
    """
    print(f'Reducing 0x{hex_value:X} = {poly_to_str(hex_value)}')
    print(f'Modulus ({modulus_name}): {poly_to_str(modulus)}  (0x{modulus:X})')
    print()
    q, r = poly_divmod(hex_value, modulus)
    print(f'Quotient  : {poly_to_str(q)}')
    print(f'Remainder : {poly_to_str(r)}  (0x{r:02X})')
    print()
    print(f'Verify: quotient * modulus + remainder = original?')
    check = 0
    # poly_mul over GF(2)
    a, b = q, modulus
    c = 0
    while a:
        if a & 1: c ^= b
        b <<= 1
        a >>= 1
    reconstructed = c ^ r
    print(f'  Reconstructed: 0x{reconstructed:X}  Match: {reconstructed == hex_value}')
    return r

# Try some examples
print('Example 1:')
reduction_explorer(0x53 * 0xCA)  # product of two AES bytes before reduction
print('---')
print('Example 2 (toy modulus):')
reduction_explorer(poly_from_coeffs(5, 2, 1), modulus=TOY_MOD, modulus_name='Toy x^3+x+1')

In [ ]:
# Section 9 — Checking irreducibility over GF(2)
# A polynomial p(x) of degree n is irreducible over GF(2) if it has no
# polynomial factors of degree 1 .. n-1 with GF(2) coefficients.

def poly_mul_gf2(a, b):
    """Multiply two polynomials over GF(2) (no reduction)."""
    result = 0
    while a:
        if a & 1: result ^= b
        b <<= 1
        a >>= 1
    return result

def is_irreducible_gf2(p):
    """Test whether p is irreducible over GF(2) by trial division."""
    deg = poly_degree(p)
    if deg <= 0:
        return False  # constants not considered
    if deg == 1:
        return True   # x and x+1 are irreducible
    # Check divisibility by all polynomials of degree 1 .. deg//2
    for d in range(1, deg // 2 + 1):
        # Enumerate all degree-d polynomials over GF(2)
        lo = 1 << d          # leading term
        hi = 1 << (d + 1)
        for candidate in range(lo, hi):
            _, r = poly_divmod(p, candidate)
            if r == 0:
                return False
    return True

# Verify the AES modulus is irreducible
print(f'AES modulus irreducible: {is_irreducible_gf2(AES_MOD)}')
print(f'Toy modulus irreducible: {is_irreducible_gf2(TOY_MOD)}')
print()

# Count all irreducible degree-8 polynomials over GF(2)
irred_8 = [v for v in range(0x100, 0x200) if is_irreducible_gf2(v)]
print(f'Number of irreducible degree-8 polynomials over GF(2): {len(irred_8)}')
print(f'First five (hex): {[hex(v) for v in irred_8[:5]]}')
print(f'AES modulus 0x{AES_MOD:03X} is in the list: {AES_MOD in irred_8}')

## Section 10 — Summary and bridge to Module 12

### Key results from this notebook

| Concept | Detail |
|---|---|
| Polynomial reduction | `poly_reduce(val, modulus)` — remainder of long division over GF(2) |
| Toy modulus | `x^3+x+1` — folding rule: `x^3 ≡ x+1` |
| AES modulus | `x^8+x^4+x^3+x+1` = `0x11B` — folding rule: `x^8 ≡ x^4+x^3+x+1` = `0x1B` |
| Irreducibility | 30 irreducible degree-8 polynomials over GF(2); AES uses `0x11B` |
| Why irreducible? | Guarantees GF(2^8) is a field: every nonzero byte has a multiplicative inverse |

### Bridge to Module 12: Multiplication in GF(2^8)

We now have all three building blocks:
1. Byte-polynomials (Module 8)
2. Polynomial multiplication over GF(2) without reduction (Module 10)
3. Polynomial reduction by the AES modulus (this module)

Module 12 combines them: **full GF(2^8) multiplication** = unreduced product, then `poly_reduce(product, 0x11B)`. That operation is at the heart of the AES MixColumns step and the xtime function.